# 502: Probability of No Further Warming Under Net-Zero Kyoto Emissions

This notebook calculates the probability that warming declines after net-zero Kyoto GHG emissions are achieved.

**Criteria for decline:**
- Peak warming occurs before 2100
- Delta warming (peak - 2100) > 0.01°C

In [1]:
import os
from pathlib import Path

import dotenv
import pandas as pd

dotenv.load_dotenv()

True

In [2]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

# Guardrails for classifying decline
PEAK_YEAR_THRESHOLD = 2100  # Peak must occur before this year
DELTA_WARMING_THRESHOLD = 0.01  # Minimum decline in °C to count as decline

In [3]:
# Input metrics file - update this with the correct manifest ID
MANIFEST_ID = "2026-05-19_09-30"
METRICS_FILE = OUTPUT_FOLDER / f"500_gsat_metrics_{MANIFEST_ID}.csv"

## Load and Filter Data

In [4]:
# Load metrics
metrics_df = pd.read_csv(METRICS_FILE)
print(f"Total rows: {len(metrics_df)}")
metrics_df.head()

Total rows: 1839600


,model,scenario,magicc_flag,run_id,gsat_peak,gsat_peak_year,gsat_2100,delta_warming
0,AIM/CGE 2.0,SSP1-19,ALL,0,1.613,2049,1.411,0.202
1,AIM/CGE 2.0,SSP1-19,ALL,1,2.593,2093,2.577,0.015
2,AIM/CGE 2.0,SSP1-19,ALL,2,1.798,2049,1.577,0.221
3,AIM/CGE 2.0,SSP1-19,ALL,3,1.380,2038,1.074,0.306
4,AIM/CGE 2.0,SSP1-19,ALL,4,1.645,2048,1.396,0.249


In [5]:
# Filter for NZKyoto scenarios only
nzkyoto_df = metrics_df[metrics_df["scenario"].str.contains("NZKyoto", na=False)].copy()
print(f"NZKyoto scenarios: {len(nzkyoto_df)} rows")
print(f"Unique scenarios: {nzkyoto_df['scenario'].nunique()}")

NZKyoto scenarios: 196800 rows
Unique scenarios: 195


In [6]:
# Filter for ALL magicc_flag (total warming, not just CO2 or GHG components)
nzkyoto_all_df = nzkyoto_df[nzkyoto_df["magicc_flag"] == "ALL"].copy()
print(f"NZKyoto ALL scenarios: {len(nzkyoto_all_df)} rows")

NZKyoto ALL scenarios: 196800 rows


## Classify Ensemble Members

In [7]:
# Apply guardrails to classify each ensemble member as "decline" or not
nzkyoto_all_df["is_decline"] = (
    (nzkyoto_all_df["gsat_peak_year"] < PEAK_YEAR_THRESHOLD) &
    (nzkyoto_all_df["delta_warming"] > DELTA_WARMING_THRESHOLD)
)

print(f"Ensemble members classified as decline: {nzkyoto_all_df['is_decline'].sum()}")
print(f"Ensemble members not classified as decline: {(~nzkyoto_all_df['is_decline']).sum()}")

Ensemble members classified as decline: 185819
Ensemble members not classified as decline: 10981


## Calculate Probability Per Scenario

In [8]:
# Group by model and scenario, calculate probability
decline_prob = nzkyoto_all_df.groupby(["model", "scenario"]).agg(
    n_ensemble=pd.NamedAgg(column="run_id", aggfunc="count"),
    n_decline=pd.NamedAgg(column="is_decline", aggfunc="sum"),
).reset_index()

# Calculate probability
decline_prob["prob_decline"] = decline_prob["n_decline"] / decline_prob["n_ensemble"]
decline_prob["prob_decline"] = decline_prob["prob_decline"].round(3)

print(f"Calculated probabilities for {len(decline_prob)} scenarios")
decline_prob.head(10)

Calculated probabilities for 328 scenarios


,model,scenario,n_ensemble,n_decline,prob_decline
0,AIM/CGE 2.0,SSP1-19_NZKyoto,600,585,0.975
1,AIM/CGE 2.0,SSP2-19_NZKyoto,600,597,0.995
2,AIM/CGE 2.1,CD-LINKS-NPi2020_400_NZKyoto,600,571,0.952
3,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional_NZKyoto,600,561,0.935
4,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutiona...,600,579,0.965
5,AIM/CGE V2.2,ENGAGE-NPi2020-1000f_NZKyoto,600,588,0.980
6,AIM/CGE V2.2,ENGAGE-NPi2020-300f_NZKyoto,600,595,0.992
7,AIM/CGE V2.2,ENGAGE-NPi2020-400f_NZKyoto,600,595,0.992
8,AIM/CGE V2.2,ENGAGE-NPi2020-500f_NZKyoto,600,595,0.992
9,AIM/CGE V2.2,ENGAGE-NPi2020-600f_NZKyoto,600,594,0.990


In [9]:
round(
    decline_prob['prob_decline'].describe(percentiles=[0.05, 0.34, 0.5, 0.67, 0.95]) * 100,
    2
)

count    32800.00
mean        94.42
std          5.10
min         72.00
5%          83.37
34%         93.84
50%         95.70
67%         97.32
95%        100.00
max        100.00
Name: prob_decline, dtype: float64

## Save Results

In [10]:
# Save the decline probability results
output_path = OUTPUT_FOLDER / f"501_nzkyoto_decline_probability_{MANIFEST_ID}.csv"
decline_prob.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

Results saved to: /Users/ganti/Documents/code/ar7_netzero_assessment/analysis/output/501_nzkyoto_decline_probability_2026-05-19_09-30.csv
